<a href="https://colab.research.google.com/github/kuds/reinforce-tactics/blob/main/notebooks/feudal_rl_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Reinforce Tactics — Feudal RL Training

This notebook trains and validates the **Feudal RL** (Manager–Worker hierarchy) agent.
It runs a short training loop, collects metrics at checkpoints, and validates that
the full pipeline works end-to-end.

## What is Feudal RL?

The Feudal RL architecture splits decision-making into two levels:

| Level | Network | Output | Update Frequency |
|-------|---------|--------|------------------|
| **Manager** | `ManagerNetwork` | Spatial goal `(x, y, type)` | Every `manager_horizon` steps |
| **Worker** | `WorkerNetwork` | Primitive action `(action_type, unit_type, from, to)` | Every step |

The **Manager** sets strategic goals (attack, defend, capture, expand) and the
**Worker** executes tactical actions conditioned on those goals. Both are trained
with PPO using separate advantage estimation.

### Checkpoints

| Checkpoint | Timesteps | Purpose |
|------------|-----------|----------|
| 1 | 2,000 | Smoke test — pipeline runs |
| 2 | 5,000 | Losses stabilize |
| 3 | 10,000 | Short eval |
| 4 | 25,000 | Convergence check |

**Runtime:** ~5–10 min on CPU, ~2–5 min on GPU (T4/L4).

---

## 1. Setup

In [ ]:
# Install dependencies
!pip install -q gymnasium stable-baselines3 sb3-contrib tensorboard numpy torch matplotlib

# Install reinforce-tactics from GitHub
!pip install -q git+https://github.com/kuds/reinforce-tactics.git

import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    print(f"  Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

In [ ]:
import time
from collections import defaultdict

import matplotlib.pyplot as plt
import numpy as np

from reinforcetactics.rl.feudal_rl import FeudalRLAgent, compute_intrinsic_reward
from reinforcetactics.rl.gym_env import StrategyGameEnv

print("Imports OK")

## 2. Configuration

Adjust these hyperparameters to experiment with training.

In [ ]:
# --- Training config ---
TOTAL_TIMESTEPS = 25_000       # Total environment steps
N_STEPS = 512                  # Steps per rollout (smaller for notebook)
BATCH_SIZE = 64                # PPO mini-batch size
N_EPOCHS = 4                   # PPO epochs per update
LEARNING_RATE = 3e-4           # Base learning rate
MANAGER_LR_SCALE = 1.0        # Manager LR = base * scale
WORKER_LR_SCALE = 1.0         # Worker LR = base * scale

# --- Feudal RL config ---
MANAGER_HORIZON = 10           # Worker steps between manager goal updates
WORKER_REWARD_ALPHA = 0.5      # Extrinsic reward weight (0=intrinsic only)

# --- PPO config ---
GAMMA = 0.99
GAE_LAMBDA = 0.95
CLIP_RANGE = 0.2
ENT_COEF = 0.05
VF_COEF = 0.5
MAX_GRAD_NORM = 0.5

# --- Environment config ---
OPPONENT = "bot"               # bot | random
MAX_STEPS = 300                # Max steps per episode

# --- Eval config ---
EVAL_EPISODES = 10             # Episodes per evaluation
CHECKPOINT_STEPS = [2_000, 5_000, 10_000, 25_000]  # Eval at these timesteps

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

print("Config loaded")
print(f"  Total timesteps: {TOTAL_TIMESTEPS:,}")
print(f"  Steps per rollout: {N_STEPS}")
print(f"  Manager horizon: {MANAGER_HORIZON}")
print(f"  Worker reward alpha: {WORKER_REWARD_ALPHA}")

## 3. Environment & Agent Setup

In [ ]:
# Create training and evaluation environments
env = StrategyGameEnv(
    map_file=None,  # Random maps
    opponent=OPPONENT,
    render_mode=None,
    max_steps=MAX_STEPS,
)

eval_env = StrategyGameEnv(
    map_file=None,
    opponent=OPPONENT,
    render_mode=None,
    max_steps=MAX_STEPS,
)

print(f"Grid: {env.grid_width}x{env.grid_height}")
print(f"Observation space: {env.observation_space}")
print(f"Action space: {env.action_space}")

In [ ]:
# Create the Feudal RL agent
agent = FeudalRLAgent(
    observation_space=env.observation_space,
    grid_width=env.grid_width,
    grid_height=env.grid_height,
    agent_player=getattr(env, "agent_player", 1),
    device=DEVICE,
)
agent.manager_horizon = MANAGER_HORIZON

# Setup training optimizers
agent.setup_training(
    learning_rate=LEARNING_RATE,
    manager_lr_scale=MANAGER_LR_SCALE,
    worker_lr_scale=WORKER_LR_SCALE,
)

# Initialize
obs, _ = env.reset(seed=SEED)
agent._last_obs = obs
agent.reset_goal()

# Count parameters
def count_params(model):
    return sum(p.numel() for p in model.parameters())

print(f"\nModel parameters:")
print(f"  Feature extractor: {count_params(agent.feature_extractor):,}")
print(f"  Manager:           {count_params(agent.manager):,}")
print(f"  Worker:            {count_params(agent.worker):,}")
total = count_params(agent.feature_extractor) + count_params(agent.manager) + count_params(agent.worker)
print(f"  Total:             {total:,}")

## 4. Sanity Checks

Before training, verify the pipeline works with a single rollout + update.

In [ ]:
print("Running sanity checks...\n")

# 1. Test select_action
test_obs = env.reset(seed=0)[0]
action, goal = agent.select_action(test_obs)
print(f"[OK] select_action -> action={action}, goal={goal}")
assert action.shape == (6,), f"Bad action shape: {action.shape}"
assert goal.shape == (3,), f"Bad goal shape: {goal.shape}"

# Reset for training
obs, _ = env.reset(seed=SEED)
agent._last_obs = obs
agent.reset_goal()

# 2. Test collect_rollout
buf = agent.collect_rollout(env, n_steps=64, gamma=GAMMA, gae_lambda=GAE_LAMBDA,
                            worker_reward_alpha=WORKER_REWARD_ALPHA)
print(f"[OK] collect_rollout -> {len(buf.w_rewards)} worker steps, "
      f"{len(buf.m_rewards)} manager segments")
assert buf.w_advantages is not None
assert len(buf.w_rewards) == 64

# 3. Test update
metrics = agent.update(buf, n_epochs=2, batch_size=32,
                       clip_range=CLIP_RANGE, ent_coef=ENT_COEF,
                       vf_coef=VF_COEF, max_grad_norm=MAX_GRAD_NORM)
print(f"[OK] update -> losses: ", end="")
for k, v in metrics.items():
    print(f"{k}={v:.4f} ", end="")
    assert np.isfinite(v), f"{k} is not finite: {v}"
print()

# 4. Test intrinsic reward
test_state = {"grid": np.zeros((env.grid_height, env.grid_width, 3), dtype=np.float32),
              "units": np.zeros((env.grid_height, env.grid_width, 3), dtype=np.float32),
              "global_features": np.zeros(6, dtype=np.float32)}
test_state["units"][5, 5, 1] = 1  # player unit
test_goal = np.array([5, 5, 0])  # attack goal at unit position
ir = compute_intrinsic_reward(test_state, test_goal, test_state, agent_player=1)
print(f"[OK] intrinsic_reward -> {ir:.2f} (unit at goal)")
assert ir >= 5.0, f"Expected >= 5.0 for unit at goal, got {ir}"

# 5. Test evaluate
eval_results = agent.evaluate(eval_env, n_episodes=3)
print(f"[OK] evaluate -> reward={eval_results['mean_reward']:.1f}, "
      f"win_rate={eval_results['win_rate']:.2f}")

print("\nAll sanity checks passed!")

## 5. Training Loop

Train the feudal agent and record metrics at checkpoints.

In [ ]:
# Reset environment and agent for fresh training
obs, _ = env.reset(seed=SEED)
agent._last_obs = obs
agent.reset_goal()

# Metrics tracking
history = defaultdict(list)
checkpoint_results = []
num_updates = TOTAL_TIMESTEPS // N_STEPS
total_timesteps = 0
start_time = time.time()

print(f"Training for {TOTAL_TIMESTEPS:,} timesteps ({num_updates} updates)")
print(f"{'='*60}")

for update_idx in range(num_updates):
    # Collect rollout
    buf = agent.collect_rollout(
        env, n_steps=N_STEPS, gamma=GAMMA, gae_lambda=GAE_LAMBDA,
        worker_reward_alpha=WORKER_REWARD_ALPHA,
    )
    total_timesteps += N_STEPS

    # PPO update
    metrics = agent.update(
        buf, n_epochs=N_EPOCHS, batch_size=BATCH_SIZE,
        clip_range=CLIP_RANGE, ent_coef=ENT_COEF,
        vf_coef=VF_COEF, max_grad_norm=MAX_GRAD_NORM,
    )

    # Record metrics
    history["timestep"].append(total_timesteps)
    for key, val in metrics.items():
        history[key].append(val)
    history["worker_mean_reward"].append(float(buf.w_rewards.mean()))
    history["worker_std_reward"].append(float(buf.w_rewards.std()))
    history["manager_segments"].append(len(buf.m_rewards))
    history["episode_dones"].append(float(buf.w_dones.sum()))

    # Track goal distribution
    if buf.has_manager_data:
        goal_types = buf.m_goals[:, 2].astype(int)
        for gt in range(4):
            history[f"goal_type_{gt}_pct"].append(
                (goal_types == gt).sum() / max(len(goal_types), 1)
            )
    else:
        for gt in range(4):
            history[f"goal_type_{gt}_pct"].append(0.0)

    # Progress logging
    if (update_idx + 1) % 5 == 0:
        elapsed = time.time() - start_time
        sps = total_timesteps / elapsed
        print(
            f"[{total_timesteps:>7,}] "
            f"w_policy={metrics['worker_policy_loss']:.3f} "
            f"m_policy={metrics['manager_policy_loss']:.3f} "
            f"w_entropy={metrics['worker_entropy']:.3f} "
            f"w_reward={buf.w_rewards.mean():.2f} "
            f"({sps:.0f} steps/s)"
        )

    # Checkpoint evaluation
    for cp_step in CHECKPOINT_STEPS:
        if total_timesteps >= cp_step and total_timesteps - N_STEPS < cp_step:
            eval_results = agent.evaluate(eval_env, n_episodes=EVAL_EPISODES)
            checkpoint_results.append({
                "timesteps": total_timesteps,
                **eval_results,
            })
            print(
                f"  >> EVAL @ {total_timesteps:,}: "
                f"reward={eval_results['mean_reward']:.1f} +/- {eval_results['std_reward']:.1f}, "
                f"win_rate={eval_results['win_rate']:.2f}"
            )

elapsed = time.time() - start_time
print(f"{'='*60}")
print(f"Training complete in {elapsed:.1f}s ({total_timesteps/elapsed:.0f} steps/s)")

## 6. Results & Visualization

In [ ]:
# Print checkpoint results table
print(f"{'Timesteps':>12} {'Mean Reward':>12} {'Std Reward':>12} {'Win Rate':>10}")
print("-" * 50)
for cp in checkpoint_results:
    print(
        f"{cp['timesteps']:>12,} "
        f"{cp['mean_reward']:>12.1f} "
        f"{cp['std_reward']:>12.1f} "
        f"{cp['win_rate']:>10.2f}"
    )

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
fig.suptitle("Feudal RL Training Metrics", fontsize=14)

timesteps = history["timestep"]

# Worker policy loss
axes[0, 0].plot(timesteps, history["worker_policy_loss"], alpha=0.7)
axes[0, 0].set_title("Worker Policy Loss")
axes[0, 0].set_xlabel("Timesteps")
axes[0, 0].grid(True, alpha=0.3)

# Manager policy loss
axes[0, 1].plot(timesteps, history["manager_policy_loss"], alpha=0.7, color="orange")
axes[0, 1].set_title("Manager Policy Loss")
axes[0, 1].set_xlabel("Timesteps")
axes[0, 1].grid(True, alpha=0.3)

# Entropy
axes[0, 2].plot(timesteps, history["worker_entropy"], label="Worker", alpha=0.7)
axes[0, 2].plot(timesteps, history["manager_entropy"], label="Manager", alpha=0.7)
axes[0, 2].set_title("Entropy")
axes[0, 2].set_xlabel("Timesteps")
axes[0, 2].legend()
axes[0, 2].grid(True, alpha=0.3)

# Worker reward
axes[1, 0].plot(timesteps, history["worker_mean_reward"], alpha=0.7, color="green")
axes[1, 0].fill_between(
    timesteps,
    np.array(history["worker_mean_reward"]) - np.array(history["worker_std_reward"]),
    np.array(history["worker_mean_reward"]) + np.array(history["worker_std_reward"]),
    alpha=0.15, color="green",
)
axes[1, 0].set_title("Worker Mean Reward (per step)")
axes[1, 0].set_xlabel("Timesteps")
axes[1, 0].grid(True, alpha=0.3)

# Value losses
axes[1, 1].plot(timesteps, history["worker_value_loss"], label="Worker", alpha=0.7)
axes[1, 1].plot(timesteps, history["manager_value_loss"], label="Manager", alpha=0.7)
axes[1, 1].set_title("Value Loss")
axes[1, 1].set_xlabel("Timesteps")
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

# Goal type distribution
goal_labels = ["Attack", "Defend", "Capture", "Expand"]
for gt in range(4):
    axes[1, 2].plot(timesteps, history[f"goal_type_{gt}_pct"],
                    label=goal_labels[gt], alpha=0.7)
axes[1, 2].set_title("Manager Goal Distribution")
axes[1, 2].set_xlabel("Timesteps")
axes[1, 2].set_ylabel("Fraction")
axes[1, 2].legend()
axes[1, 2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
if checkpoint_results:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    fig.suptitle("Evaluation Results at Checkpoints", fontsize=14)

    cp_steps = [cp["timesteps"] for cp in checkpoint_results]
    cp_rewards = [cp["mean_reward"] for cp in checkpoint_results]
    cp_stds = [cp["std_reward"] for cp in checkpoint_results]
    cp_winrates = [cp["win_rate"] for cp in checkpoint_results]

    ax1.errorbar(cp_steps, cp_rewards, yerr=cp_stds, marker="o", capsize=5)
    ax1.set_title("Mean Episode Reward")
    ax1.set_xlabel("Timesteps")
    ax1.set_ylabel("Reward")
    ax1.grid(True, alpha=0.3)

    ax2.bar(range(len(cp_steps)), cp_winrates, tick_label=[f"{s:,}" for s in cp_steps])
    ax2.set_title("Win Rate vs SimpleBot")
    ax2.set_xlabel("Timesteps")
    ax2.set_ylabel("Win Rate")
    ax2.set_ylim(0, 1)
    ax2.grid(True, alpha=0.3, axis="y")

    plt.tight_layout()
    plt.show()
else:
    print("No checkpoint results to plot.")

## 7. Pipeline Validation

Verify that the training pipeline produces expected outputs and the model
can be saved/loaded correctly.

In [ ]:
import tempfile
from pathlib import Path

print("Running validation checks...\n")
all_passed = True

# 1. Training produced finite losses
for key in ["worker_policy_loss", "manager_policy_loss", "worker_entropy"]:
    vals = history[key]
    if all(np.isfinite(v) for v in vals):
        print(f"  [PASS] {key}: all finite (last={vals[-1]:.4f})")
    else:
        print(f"  [FAIL] {key}: contains non-finite values")
        all_passed = False

# 2. Worker entropy stays positive (exploration)
if all(v > 0 for v in history["worker_entropy"]):
    print(f"  [PASS] Worker entropy always positive (min={min(history['worker_entropy']):.4f})")
else:
    print(f"  [FAIL] Worker entropy went to zero")
    all_passed = False

# 3. Manager segments were created
total_segments = sum(history["manager_segments"])
if total_segments > 0:
    print(f"  [PASS] Manager produced {total_segments} goal segments")
else:
    print(f"  [FAIL] No manager segments created")
    all_passed = False

# 4. Goal type diversity
last_goal_pcts = [history[f"goal_type_{gt}_pct"][-1] for gt in range(4)]
nonzero_types = sum(1 for p in last_goal_pcts if p > 0)
if nonzero_types >= 2:
    print(f"  [PASS] Manager uses {nonzero_types}/4 goal types (diverse goals)")
else:
    print(f"  [WARN] Manager only uses {nonzero_types}/4 goal types (may need more training)")

# 5. Checkpoint save/load roundtrip
with tempfile.TemporaryDirectory() as tmpdir:
    ckpt_path = str(Path(tmpdir) / "feudal_test.pt")
    agent.save_checkpoint(ckpt_path)

    agent2 = FeudalRLAgent(
        observation_space=env.observation_space,
        grid_width=env.grid_width,
        grid_height=env.grid_height,
        device=DEVICE,
    )
    agent2.load_checkpoint(ckpt_path)

    # Verify weights match
    match = all(
        torch.allclose(p1, p2)
        for p1, p2 in zip(agent.manager.parameters(), agent2.manager.parameters())
    )
    if match:
        print(f"  [PASS] Checkpoint save/load roundtrip")
    else:
        print(f"  [FAIL] Checkpoint weights don't match after load")
        all_passed = False

    # Verify loaded model can act
    test_obs = env.reset(seed=0)[0]
    action, goal = agent2.select_action(test_obs)
    if action.shape == (6,):
        print(f"  [PASS] Loaded model produces valid actions")
    else:
        print(f"  [FAIL] Loaded model action shape: {action.shape}")
        all_passed = False

# 6. Episodes completed during training
total_dones = sum(history["episode_dones"])
if total_dones > 0:
    print(f"  [PASS] Completed {int(total_dones)} episodes during training")
else:
    print(f"  [WARN] No episodes completed (may need longer training or shorter episodes)")

print()
if all_passed:
    print("All validation checks PASSED!")
else:
    print("Some validation checks FAILED - review output above.")

## 8. Watch the Agent Play

Run a single game and display the manager's goal decisions.

In [ ]:
goal_names = {0: "Attack", 1: "Defend", 2: "Capture", 3: "Expand"}
action_names = {
    0: "Create", 1: "Move", 2: "Attack", 3: "Seize", 4: "Heal",
    5: "EndTurn", 6: "Paralyze", 7: "Haste", 8: "DefBuff", 9: "AtkBuff",
}

watch_env = StrategyGameEnv(map_file=None, opponent=OPPONENT, render_mode=None, max_steps=MAX_STEPS)
obs, _ = watch_env.reset(seed=99)
agent.reset_goal()

done = False
step = 0
ep_reward = 0.0
goal_history = []

while not done and step < 100:  # Cap display at 100 steps
    action, goal = agent.select_action(obs, deterministic=True)
    obs, reward, terminated, truncated, info = watch_env.step(action)
    done = terminated or truncated
    ep_reward += reward
    step += 1

    goal_type = int(goal[2])
    action_type = int(action[0])
    goal_history.append(goal_type)

    if step <= 20 or done:  # Show first 20 steps + final
        valid = info.get("valid_action", "?")
        print(
            f"  Step {step:3d} | "
            f"Goal: {goal_names.get(goal_type, '?'):8s} ({int(goal[0]):2d},{int(goal[1]):2d}) | "
            f"Action: {action_names.get(action_type, '?'):8s} | "
            f"Valid: {valid} | Reward: {reward:+.1f}"
        )

if step > 20 and not done:
    print(f"  ... ({step - 20} more steps)")

print(f"\nGame ended at step {step}. Total reward: {ep_reward:.1f}")
winner = info.get("winner")
if winner == 1:
    print("Result: Agent WON")
elif winner is not None:
    print(f"Result: Agent LOST (winner: player {winner})")
else:
    print("Result: Draw / Truncated")

# Goal type usage summary
print(f"\nGoal type distribution across game:")
for gt in range(4):
    count = sum(1 for g in goal_history if g == gt)
    pct = count / max(len(goal_history), 1) * 100
    print(f"  {goal_names[gt]:8s}: {count:3d} ({pct:.0f}%)")

## 9. Next Steps

This notebook validated the feudal RL pipeline end-to-end. For full training:

1. **Scale up**: Use `train/train_feudal_rl.py` with `--total-timesteps 10000000`
2. **Tune the manager horizon**: Try `--manager-horizon 5` (faster goals) or `20` (longer-term strategy)
3. **Adjust reward balance**: `--worker-reward-alpha 0.3` emphasizes intrinsic rewards
4. **Compare with flat PPO**: Run `--mode flat --use-action-masking` for a baseline

```bash
# Full training example
python train/train_feudal_rl.py \
    --mode feudal \
    --total-timesteps 5000000 \
    --manager-horizon 10 \
    --worker-reward-alpha 0.5 \
    --opponent bot \
    --device auto
```